In [1]:
import numpy as np

# Timeseries Data Preparation

## 1 Utility Functions

### `generator` function from Notebook 2B

In [2]:
def generator(
    data, target_col, lookback, delay, step, min_index, max_index, shuffle=False,
    batch_size=128):
    """Generate the sample and target from the given timeseries data

    Parameters
    ----------
    data : array-like
        Timeseries data with all the features to be used as predictors.
        Has an expected shape of num_rows x num_features.

    target_col : int
        Index of the column in `data` which would be used as the target

    lookback : int
        How far back in history do we use to construct the sample

    delay : int
        Time difference between target and sample. A zero delay
        corresponds to the next timestep wrt the sample.

    step : int
        Time step between values within the sample

    min_index : int
        Minimum index to be considered when constructing the sample.
        Useful for separating train, validation, and test datasets.

    max_index : int
        Maximum index to be considered when constructing the sample.
        Useful for separating train, validation, and test datasets.

    shuffle : bool
        Whether to shuffle the resulting sample and target pairs.

    batch_size : int
        The batch size

    Yields
    ------
    samples, target : array-like
        For each iteration, this function yields the samples and target
        pairs to be used as predictors and targets. Samples has a shape
        of `batch_size` x `num_features` meanwhile target has a shape of
        `batch_size`
    """

    # Set default max_index if not provided
    if max_index is None:
        max_index = len(data) - delay - 1

    # Start index for data sampling
    i = min_index + lookback

    # Infinite loop to generate samples indefinitely
    while True:
        # Shuffle data if required; otherwise, generate sequential batches
        if shuffle:
            rows = np.random.randint(min_index + lookback, max_index, size=batch_size)
        else:
            if i + batch_size >= max_index:
                i = min_index + lookback  # Reset index if it goes beyond the max index
            rows = np.arange(i, min(i + batch_size, max_index))
            i += len(rows)

        # Initialize numpy arrays to store samples and targets
        samples = np.zeros((len(rows), lookback // step, data.shape[-1]))
        targets = np.zeros((len(rows),))

        # Populate samples and targets arrays
        for j, row in enumerate(rows):
            indices = range(rows[j] - lookback, rows[j], step)  # Select indices for sample
            samples[j] = data[indices]  # Assign the sliced data to samples
            targets[j] = data[rows[j] + delay][target_col]  # Set the corresponding target

        # Yield a batch of samples and targets
        yield samples, targets

In [3]:
help(generator)

Help on function generator in module __main__:

generator(data, target_col, lookback, delay, step, min_index, max_index, shuffle=False, batch_size=128)
    Generate the sample and target from the given timeseries data
    
    Parameters
    ----------
    data : array-like
        Timeseries data with all the features to be used as predictors.
        Has an expected shape of num_rows x num_features.
    
    target_col : int
        Index of the column in `data` which would be used as the target
    
    lookback : int
        How far back in history do we use to construct the sample
    
    delay : int
        Time difference between target and sample. A zero delay
        corresponds to the next timestep wrt the sample.
    
    step : int
        Time step between values within the sample
    
    min_index : int
        Minimum index to be considered when constructing the sample.
        Useful for separating train, validation, and test datasets.
    
    max_index : int
       

### From Keras Utils `timeseries_dataset_from_array()`

In [4]:
from tensorflow import keras

In [5]:
help(keras.utils.timeseries_dataset_from_array)

Help on function timeseries_dataset_from_array in module keras.src.utils.timeseries_dataset:

timeseries_dataset_from_array(data, targets, sequence_length, sequence_stride=1, sampling_rate=1, batch_size=128, shuffle=False, seed=None, start_index=None, end_index=None)
    Creates a dataset of sliding windows over a timeseries provided as array.
    
    This function takes in a sequence of data-points gathered at
    equal intervals, along with time series parameters such as
    length of the sequences/windows, spacing between two sequence/windows, etc.,
    to produce batches of timeseries inputs and targets.
    
    Args:
      data: Numpy array or eager tensor
        containing consecutive data points (timesteps).
        Axis 0 is expected to be the time dimension.
      targets: Targets corresponding to timesteps in `data`.
        `targets[i]` should be the target
        corresponding to the window that starts at index `i`
        (see example 2 below).
        Pass None if you

## 2 Examples Using `numpy` Indices

In [6]:
data = np.arange(0, 15, 1)[:, np.newaxis]
data

array([[ 0],
       [ 1],
       [ 2],
       [ 3],
       [ 4],
       [ 5],
       [ 6],
       [ 7],
       [ 8],
       [ 9],
       [10],
       [11],
       [12],
       [13],
       [14]])

In [7]:
data.shape

(15, 1)

### `generator`

**Parameter Settings**

```
Lookback: 7
Delay: 4
Step: 1
```

In [8]:
lookback = 7
delay = 4
step = 1
min_index = 0
max_index = 14
batch_size = 1
shuffle = False

ts_dataset = generator(data, 0, lookback, delay, step, min_index, max_index,
                       shuffle, batch_size)

In [9]:
i = 1

for samples, target in ts_dataset:
    print(f'Sample-Target Pair {i}')
    print(f'Sample: {samples}')
    print(f'Target: {target}\n')
    if i == 3:
        break
    i += 1

Sample-Target Pair 1
Sample: [[[0.]
  [1.]
  [2.]
  [3.]
  [4.]
  [5.]
  [6.]]]
Target: [11.]

Sample-Target Pair 2
Sample: [[[1.]
  [2.]
  [3.]
  [4.]
  [5.]
  [6.]
  [7.]]]
Target: [12.]

Sample-Target Pair 3
Sample: [[[2.]
  [3.]
  [4.]
  [5.]
  [6.]
  [7.]
  [8.]]]
Target: [13.]



**Parameter Settings**

```
Lookback: 6
Delay: 3
Step: 2
```

In [10]:
lookback = 6
delay = 3
step = 2
min_index = 0
max_index = 14
batch_size = 1
shuffle = False

ts_dataset = generator(data, 0, lookback, delay, step, min_index, max_index,
                       shuffle, batch_size)

In [11]:
i = 1

for samples, target in ts_dataset:
    print(f'Sample-Target Pair {i}')
    print(f'Sample: {samples}')
    print(f'Target: {target}\n')
    if i == 3:
        break
    i += 1

Sample-Target Pair 1
Sample: [[[0.]
  [2.]
  [4.]]]
Target: [9.]

Sample-Target Pair 2
Sample: [[[1.]
  [3.]
  [5.]]]
Target: [10.]

Sample-Target Pair 3
Sample: [[[2.]
  [4.]
  [6.]]]
Target: [11.]



### `timeseries_dataset_from_array()`

**Parameter Settings**

```
Ahead: 5
Sequence Length: 7
Sampling Rate: 1
Sequence Stride: 1
```

In [12]:
ahead = 5
sampling_rate = 1
sequence_length = 7
sequence_stride = 1
target_delay = sampling_rate * (sequence_length + ahead - 1)
last_predictor_index = sampling_rate * (ahead)
batch_size = 1
shuffle = False

In [13]:
ts_dataset = keras.utils.timeseries_dataset_from_array(
    data[:-last_predictor_index],
    targets=data[:, 0][target_delay:],
    sampling_rate=sampling_rate,
    sequence_length=sequence_length,
    sequence_stride=sequence_stride,
    shuffle=shuffle,
    batch_size=batch_size
)

In [14]:
i = 1

for samples, target in ts_dataset:
    print(f'Sample-Target Pair {i}')
    print(f'Sample: {samples}')
    print(f'Target: {target}\n')
    if i == 3:
        break
    i += 1

Sample-Target Pair 1
Sample: [[[0]
  [1]
  [2]
  [3]
  [4]
  [5]
  [6]]]
Target: [11]

Sample-Target Pair 2
Sample: [[[1]
  [2]
  [3]
  [4]
  [5]
  [6]
  [7]]]
Target: [12]

Sample-Target Pair 3
Sample: [[[2]
  [3]
  [4]
  [5]
  [6]
  [7]
  [8]]]
Target: [13]



**Parameter Settings**

```
Ahead: 2.5
Sequence Length: 3
Sampling Rate: 2
Sequence Stride: 1
```

In [15]:
ahead = 2.5
sampling_rate = 2
sequence_length = 3
sequence_stride = 1
target_delay = int(sampling_rate * (sequence_length + ahead - 1))
last_predictor_index = int(sampling_rate * (ahead))
batch_size = 1
shuffle = False

In [16]:
ts_dataset = keras.utils.timeseries_dataset_from_array(
    data[:-last_predictor_index],
    targets=data[:, 0][target_delay:],
    sampling_rate=sampling_rate,
    sequence_length=sequence_length,
    sequence_stride=sequence_stride,
    shuffle=shuffle,
    batch_size=batch_size
)

In [17]:
i = 1

for samples, target in ts_dataset:
    print(f'Sample-Target Pair {i}')
    print(f'Sample: {samples}')
    print(f'Target: {target}\n')
    if i == 3:
        break
    i += 1

Sample-Target Pair 1
Sample: [[[0]
  [2]
  [4]]]
Target: [9]

Sample-Target Pair 2
Sample: [[[1]
  [3]
  [5]]]
Target: [10]

Sample-Target Pair 3
Sample: [[[2]
  [4]
  [6]]]
Target: [11]



**Parameter Settings**

```
Ahead: 4
Sequence Length: 3
Sampling Rate: 2
Sequence Stride: 1
```

In [18]:
ahead = 4
sequence_length = 3
sampling_rate = 2
sequence_stride = 1
target_delay = int(sampling_rate * (sequence_length + ahead - 1))
last_predictor_index = int(sampling_rate * (ahead))
batch_size = 1
shuffle = False

In [19]:
ts_dataset = keras.utils.timeseries_dataset_from_array(
    data[:-last_predictor_index],
    targets=data[:, 0][target_delay:],
    sampling_rate=sampling_rate,
    sequence_length=sequence_length,
    sequence_stride=sequence_stride,
    shuffle=shuffle,
    batch_size=batch_size
)

In [20]:
i = 1

for samples, target in ts_dataset:
    print(f'Sample-Target Pair {i}')
    print(f'Sample: {samples}')
    print(f'Target: {target}\n')
    if i == 3:
        break
    i += 1

Sample-Target Pair 1
Sample: [[[0]
  [2]
  [4]]]
Target: [12]

Sample-Target Pair 2
Sample: [[[1]
  [3]
  [5]]]
Target: [13]

Sample-Target Pair 3
Sample: [[[2]
  [4]
  [6]]]
Target: [14]



**Parameter Settings**

```
Ahead: 4
Sequence Length: 3
Sampling Rate: 2
Sequence Stride: 2
```

In [21]:
ahead = 4
sequence_length = 3
sampling_rate = 2
sequence_stride = 2
target_delay = int(sampling_rate * (sequence_length + ahead - 1))
last_predictor_index = int(sampling_rate * (ahead))
batch_size = 1
shuffle = False

In [22]:
ts_dataset = keras.utils.timeseries_dataset_from_array(
    data[:-last_predictor_index],
    targets=data[:, 0][target_delay:],
    sampling_rate=sampling_rate,
    sequence_length=sequence_length,
    sequence_stride=sequence_stride,
    shuffle=shuffle,
    batch_size=batch_size
)

In [23]:
i = 1

for samples, target in ts_dataset:
    print(f'Sample-Target Pair {i}')
    print(f'Sample: {samples}')
    print(f'Target: {target}\n')
    if i == 3:
        break
    i += 1

Sample-Target Pair 1
Sample: [[[0]
  [2]
  [4]]]
Target: [12]

Sample-Target Pair 2
Sample: [[[2]
  [4]
  [6]]]
Target: [14]



## 3 Examples Using Hourly Energy Consumption

In [24]:
from glob import glob

import pandas as pd

### Data Loading and Preprocessing

In [25]:
# Load datasets
data = pd.DataFrame()
for hourly_data in glob('data/*.csv'):
    cur_data = pd.read_csv(hourly_data, parse_dates=['Datetime']).set_index('Datetime')
    data = cur_data.join(data)

# Drop nulls
data = data.dropna()

# Make index consistent
include_dates = pd.date_range(data.index[0], data.index[-1], freq='h')
data = pd.DataFrame(index=include_dates).join(data)
data.index = data.index.set_names('Datetime')

# Impute nulls
data = data.fillna(data.median())

In [26]:
data.head()

,AEP_MW,PJMW_MW,PJME_MW
Datetime,,,
2004-10-01 01:00:00,12379.0,4628.0,24025.0
2004-10-01 02:00:00,11935.0,4520.0,22845.0
2004-10-01 03:00:00,11692.0,4431.0,22138.0
2004-10-01 04:00:00,11597.0,4383.0,21922.0
2004-10-01 05:00:00,11681.0,4460.0,22193.0


### Scenario 1

Use the hourly energy consumption of `AEP`, `PJMW`, and `PJME` to predict the total daily consumption of `PJMW` one month in advance using 2 weeks worth of hourly consumption data.

#### Using the `generator` function

In [27]:
# Note data is in hourly granularity, thus we compute each parameter accordingly
lookback = 14 * 24 # 14 days, 24 hour each day
delay = 28 * 24 - 1 # 28 days, 24 hour each day, less 1 hour (due to index implementation)
step = 1 # We use hourly consumption
min_index = 0
max_index = len(data)
batch_size = 1
shuffle = False

In [28]:
ts_dataset = generator(data.to_numpy(), 1, lookback, delay, step, min_index,
                       max_index, shuffle, batch_size)

In [29]:
for sample, target in ts_dataset:
    print(sample)
    print(target)
    break

[[[12379.  4628. 24025.]
  [11935.  4520. 22845.]
  [11692.  4431. 22138.]
  ...
  [15464.  5740. 31198.]
  [14544.  5374. 28432.]
  [13564.  4994. 25760.]]]
[5160.]


In [30]:
sample.shape

(1, 336, 3)

In [31]:
14*24

336

In [32]:
indices = np.arange(len(data))[:, np.newaxis]

In [33]:
ts_dataset = generator(indices, 0, lookback, delay, step, min_index,
                       max_index, shuffle, batch_size)

In [34]:
for sample_indices, target_indices in ts_dataset:
    print(sample_indices)
    print(target_indices)
    break

[[[  0.]
  [  1.]
  [  2.]
  [  3.]
  [  4.]
  [  5.]
  [  6.]
  [  7.]
  [  8.]
  [  9.]
  [ 10.]
  [ 11.]
  [ 12.]
  [ 13.]
  [ 14.]
  [ 15.]
  [ 16.]
  [ 17.]
  [ 18.]
  [ 19.]
  [ 20.]
  [ 21.]
  [ 22.]
  [ 23.]
  [ 24.]
  [ 25.]
  [ 26.]
  [ 27.]
  [ 28.]
  [ 29.]
  [ 30.]
  [ 31.]
  [ 32.]
  [ 33.]
  [ 34.]
  [ 35.]
  [ 36.]
  [ 37.]
  [ 38.]
  [ 39.]
  [ 40.]
  [ 41.]
  [ 42.]
  [ 43.]
  [ 44.]
  [ 45.]
  [ 46.]
  [ 47.]
  [ 48.]
  [ 49.]
  [ 50.]
  [ 51.]
  [ 52.]
  [ 53.]
  [ 54.]
  [ 55.]
  [ 56.]
  [ 57.]
  [ 58.]
  [ 59.]
  [ 60.]
  [ 61.]
  [ 62.]
  [ 63.]
  [ 64.]
  [ 65.]
  [ 66.]
  [ 67.]
  [ 68.]
  [ 69.]
  [ 70.]
  [ 71.]
  [ 72.]
  [ 73.]
  [ 74.]
  [ 75.]
  [ 76.]
  [ 77.]
  [ 78.]
  [ 79.]
  [ 80.]
  [ 81.]
  [ 82.]
  [ 83.]
  [ 84.]
  [ 85.]
  [ 86.]
  [ 87.]
  [ 88.]
  [ 89.]
  [ 90.]
  [ 91.]
  [ 92.]
  [ 93.]
  [ 94.]
  [ 95.]
  [ 96.]
  [ 97.]
  [ 98.]
  [ 99.]
  [100.]
  [101.]
  [102.]
  [103.]
  [104.]
  [105.]
  [106.]
  [107.]
  [108.]
  [109.]
  [110.]
 

In [35]:
data.iloc[target_indices]

,AEP_MW,PJMW_MW,PJME_MW
Datetime,,,
2004-11-12,14636.0,5160.0,27017.0


In [36]:
data.iloc[sample_indices.flatten()]

,AEP_MW,PJMW_MW,PJME_MW
Datetime,,,
2004-10-01 01:00:00,12379.0,4628.0,24025.0
2004-10-01 02:00:00,11935.0,4520.0,22845.0
2004-10-01 03:00:00,11692.0,4431.0,22138.0
2004-10-01 04:00:00,11597.0,4383.0,21922.0
2004-10-01 05:00:00,11681.0,4460.0,22193.0
...,...,...,...
2004-10-14 20:00:00,16129.0,6092.0,33567.0
2004-10-14 21:00:00,15940.0,6025.0,32747.0
2004-10-14 22:00:00,15464.0,5740.0,31198.0


In [37]:
data.iloc[target_indices].index[0] - data.iloc[sample_indices.flatten()].index[-1]

Timedelta('28 days 00:00:00')

#### Using `timeseries_dataset_from_array()`

In [38]:
ahead = 28 * 24 # 28 days, 24 hours in a day
sequence_length = 14 * 24 # 14 days, 24 hours in a day
sampling_rate = 1
sequence_stride = 1
target_delay = int(sampling_rate * (sequence_length + ahead - 1))
last_predictor_index = int(sampling_rate * (ahead))
batch_size = 1
shuffle = False

In [39]:
ts_dataset = keras.utils.timeseries_dataset_from_array(
    data.to_numpy()[:-last_predictor_index],
    targets=data.to_numpy()[:, 1][target_delay:],
    sampling_rate=sampling_rate,
    sequence_length=sequence_length,
    sequence_stride=sequence_stride,
    shuffle=shuffle,
    batch_size=batch_size
)

In [40]:
for sample, target in ts_dataset:
    print(sample)
    print(target)
    break

tf.Tensor(
[[[12379.  4628. 24025.]
  [11935.  4520. 22845.]
  [11692.  4431. 22138.]
  ...
  [15464.  5740. 31198.]
  [14544.  5374. 28432.]
  [13564.  4994. 25760.]]], shape=(1, 336, 3), dtype=float64)
tf.Tensor([5160.], shape=(1,), dtype=float64)


In [41]:
indices = np.arange(len(data))[:, np.newaxis]

In [42]:
ts_dataset = keras.utils.timeseries_dataset_from_array(
    indices[:-last_predictor_index],
    targets=indices[:, 0][target_delay:],
    sampling_rate=sampling_rate,
    sequence_length=sequence_length,
    sequence_stride=sequence_stride,
    shuffle=shuffle,
    batch_size=batch_size
)

In [43]:
for sample, target in ts_dataset:
    print(sample)
    print(target)
    break

tf.Tensor(
[[[  0]
  [  1]
  [  2]
  [  3]
  [  4]
  [  5]
  [  6]
  [  7]
  [  8]
  [  9]
  [ 10]
  [ 11]
  [ 12]
  [ 13]
  [ 14]
  [ 15]
  [ 16]
  [ 17]
  [ 18]
  [ 19]
  [ 20]
  [ 21]
  [ 22]
  [ 23]
  [ 24]
  [ 25]
  [ 26]
  [ 27]
  [ 28]
  [ 29]
  [ 30]
  [ 31]
  [ 32]
  [ 33]
  [ 34]
  [ 35]
  [ 36]
  [ 37]
  [ 38]
  [ 39]
  [ 40]
  [ 41]
  [ 42]
  [ 43]
  [ 44]
  [ 45]
  [ 46]
  [ 47]
  [ 48]
  [ 49]
  [ 50]
  [ 51]
  [ 52]
  [ 53]
  [ 54]
  [ 55]
  [ 56]
  [ 57]
  [ 58]
  [ 59]
  [ 60]
  [ 61]
  [ 62]
  [ 63]
  [ 64]
  [ 65]
  [ 66]
  [ 67]
  [ 68]
  [ 69]
  [ 70]
  [ 71]
  [ 72]
  [ 73]
  [ 74]
  [ 75]
  [ 76]
  [ 77]
  [ 78]
  [ 79]
  [ 80]
  [ 81]
  [ 82]
  [ 83]
  [ 84]
  [ 85]
  [ 86]
  [ 87]
  [ 88]
  [ 89]
  [ 90]
  [ 91]
  [ 92]
  [ 93]
  [ 94]
  [ 95]
  [ 96]
  [ 97]
  [ 98]
  [ 99]
  [100]
  [101]
  [102]
  [103]
  [104]
  [105]
  [106]
  [107]
  [108]
  [109]
  [110]
  [111]
  [112]
  [113]
  [114]
  [115]
  [116]
  [117]
  [118]
  [119]
  [120]
  [121]
  [122]
  [12

### Scenario 2

Use the 6-hourly energy consumption of `AEP`, `PJMW`, and `PJME` to predict the total daily consumption of `PJMW` one month in advance using one month worth of hourly consumption data.

#### Using the `generator` function

In [44]:
# Note data is in hourly granularity, thus we compute each parameter accordingly
lookback = 28 * 24 # 28 days, 24 hour each day
delay = 28 * 24 - 6 # 28 days, 24 hour each day, less 6 hours (due to index implementation)
step = 6 # We use 6-hourly consumption
min_index = 0
max_index = len(data)
batch_size = 1
shuffle = False

In [45]:
ts_dataset = generator(data.to_numpy(), 1, lookback, delay, step, min_index,
                       max_index, shuffle, batch_size)

In [46]:
for sample, target in ts_dataset:
    print(sample)
    print(target)
    break

[[[12379.  4628. 24025.]
  [13692.  5360. 27487.]
  [15404.  5783. 33157.]
  [15034.  5714. 32348.]
  [12260.  4578. 23953.]
  [11866.  4499. 23360.]
  [14056.  5413. 29645.]
  [13379.  5362. 30006.]
  [11443.  4329. 23723.]
  [10795.  4124. 21500.]
  [12997.  4973. 27279.]
  [12979.  5151. 28522.]
  [11817.  4375. 22469.]
  [13862.  5275. 27496.]
  [15662.  5746. 32953.]
  [15368.  5754. 33275.]
  [12532.  4597. 24023.]
  [14393.  5462. 27708.]
  [15423.  5643. 31341.]
  [14976.  5669. 31537.]
  [12764.  4803. 23492.]
  [14677.  5757. 28699.]
  [15137.  5606. 31599.]
  [14765.  5660. 31812.]
  [12484.  4710. 23573.]
  [14394.  5674. 28427.]
  [15508.  5765. 32420.]
  [15042.  5798. 32864.]
  [12468.  4626. 23541.]
  [14209.  5465. 27680.]
  [15626.  5814. 33178.]
  [14930.  5613. 32428.]
  [12214.  4528. 23672.]
  [11733.  4567. 23071.]
  [13818.  5264. 28645.]
  [13557.  5257. 29053.]
  [11269.  4279. 22839.]
  [10862.  4100. 21449.]
  [12931.  5021. 27108.]
  [13247.  5305. 28329.]


In [47]:
sample.shape

(1, 112, 3)

In [48]:
28*24 / 6

112.0

In [49]:
indices = np.arange(len(data))[:, np.newaxis]

In [50]:
ts_dataset = generator(indices, 0, lookback, delay, step, min_index,
                       max_index, shuffle, batch_size)

In [51]:
for sample_indices, target_indices in ts_dataset:
    print(sample_indices)
    print(target_indices)
    break

[[[  0.]
  [  6.]
  [ 12.]
  [ 18.]
  [ 24.]
  [ 30.]
  [ 36.]
  [ 42.]
  [ 48.]
  [ 54.]
  [ 60.]
  [ 66.]
  [ 72.]
  [ 78.]
  [ 84.]
  [ 90.]
  [ 96.]
  [102.]
  [108.]
  [114.]
  [120.]
  [126.]
  [132.]
  [138.]
  [144.]
  [150.]
  [156.]
  [162.]
  [168.]
  [174.]
  [180.]
  [186.]
  [192.]
  [198.]
  [204.]
  [210.]
  [216.]
  [222.]
  [228.]
  [234.]
  [240.]
  [246.]
  [252.]
  [258.]
  [264.]
  [270.]
  [276.]
  [282.]
  [288.]
  [294.]
  [300.]
  [306.]
  [312.]
  [318.]
  [324.]
  [330.]
  [336.]
  [342.]
  [348.]
  [354.]
  [360.]
  [366.]
  [372.]
  [378.]
  [384.]
  [390.]
  [396.]
  [402.]
  [408.]
  [414.]
  [420.]
  [426.]
  [432.]
  [438.]
  [444.]
  [450.]
  [456.]
  [462.]
  [468.]
  [474.]
  [480.]
  [486.]
  [492.]
  [498.]
  [504.]
  [510.]
  [516.]
  [522.]
  [528.]
  [534.]
  [540.]
  [546.]
  [552.]
  [558.]
  [564.]
  [570.]
  [576.]
  [582.]
  [588.]
  [594.]
  [600.]
  [606.]
  [612.]
  [618.]
  [624.]
  [630.]
  [636.]
  [642.]
  [648.]
  [654.]
  [660.]
 

In [52]:
data.iloc[target_indices]

,AEP_MW,PJMW_MW,PJME_MW
Datetime,,,
2004-11-25 19:00:00,14104.0,5503.0,27990.0


In [53]:
data.iloc[sample_indices.flatten()]

,AEP_MW,PJMW_MW,PJME_MW
Datetime,,,
2004-10-01 01:00:00,12379.0,4628.0,24025.0
2004-10-01 07:00:00,13692.0,5360.0,27487.0
2004-10-01 13:00:00,15404.0,5783.0,33157.0
2004-10-01 19:00:00,15034.0,5714.0,32348.0
2004-10-02 01:00:00,12260.0,4578.0,23953.0
...,...,...,...
2004-10-27 19:00:00,15453.0,5962.0,33488.0
2004-10-28 01:00:00,12604.0,4837.0,24362.0
2004-10-28 07:00:00,14134.0,5612.0,29478.0


In [54]:
data.iloc[target_indices].index[0] - data.iloc[sample_indices.flatten()].index[-1]

Timedelta('28 days 00:00:00')

#### Using `timeseries_dataset_from_array()`

In [55]:
ahead = 28 * 24 / 6 # 28 days, 24 hours in a day, divided by sampling rate
sequence_length = 28 * 24 / 6 # 14 days, 24 hours in a day, sampled every 6 hours
sampling_rate = 6
sequence_stride = 1
target_delay = int(sampling_rate * (sequence_length + ahead - 1))
last_predictor_index = int(sampling_rate * (ahead))
batch_size = 1
shuffle = False

In [56]:
ts_dataset = keras.utils.timeseries_dataset_from_array(
    data.to_numpy()[:-last_predictor_index],
    targets=data.to_numpy()[:, 1][target_delay:],
    sampling_rate=sampling_rate,
    sequence_length=sequence_length,
    sequence_stride=sequence_stride,
    shuffle=shuffle,
    batch_size=batch_size
)

In [57]:
for sample, target in ts_dataset:
    print(sample)
    print(target)
    break

tf.Tensor(
[[[12379.  4628. 24025.]
  [13692.  5360. 27487.]
  [15404.  5783. 33157.]
  [15034.  5714. 32348.]
  [12260.  4578. 23953.]
  [11866.  4499. 23360.]
  [14056.  5413. 29645.]
  [13379.  5362. 30006.]
  [11443.  4329. 23723.]
  [10795.  4124. 21500.]
  [12997.  4973. 27279.]
  [12979.  5151. 28522.]
  [11817.  4375. 22469.]
  [13862.  5275. 27496.]
  [15662.  5746. 32953.]
  [15368.  5754. 33275.]
  [12532.  4597. 24023.]
  [14393.  5462. 27708.]
  [15423.  5643. 31341.]
  [14976.  5669. 31537.]
  [12764.  4803. 23492.]
  [14677.  5757. 28699.]
  [15137.  5606. 31599.]
  [14765.  5660. 31812.]
  [12484.  4710. 23573.]
  [14394.  5674. 28427.]
  [15508.  5765. 32420.]
  [15042.  5798. 32864.]
  [12468.  4626. 23541.]
  [14209.  5465. 27680.]
  [15626.  5814. 33178.]
  [14930.  5613. 32428.]
  [12214.  4528. 23672.]
  [11733.  4567. 23071.]
  [13818.  5264. 28645.]
  [13557.  5257. 29053.]
  [11269.  4279. 22839.]
  [10862.  4100. 21449.]
  [12931.  5021. 27108.]
  [13247.  530

In [58]:
indices = np.arange(len(data))[:, np.newaxis]

In [59]:
ts_dataset = keras.utils.timeseries_dataset_from_array(
    indices[:-last_predictor_index],
    targets=indices[:, 0][target_delay:],
    sampling_rate=sampling_rate,
    sequence_length=sequence_length,
    sequence_stride=sequence_stride,
    shuffle=shuffle,
    batch_size=batch_size
)

In [60]:
for sample, target in ts_dataset:
    print(sample)
    print(target)
    break

tf.Tensor(
[[[  0]
  [  6]
  [ 12]
  [ 18]
  [ 24]
  [ 30]
  [ 36]
  [ 42]
  [ 48]
  [ 54]
  [ 60]
  [ 66]
  [ 72]
  [ 78]
  [ 84]
  [ 90]
  [ 96]
  [102]
  [108]
  [114]
  [120]
  [126]
  [132]
  [138]
  [144]
  [150]
  [156]
  [162]
  [168]
  [174]
  [180]
  [186]
  [192]
  [198]
  [204]
  [210]
  [216]
  [222]
  [228]
  [234]
  [240]
  [246]
  [252]
  [258]
  [264]
  [270]
  [276]
  [282]
  [288]
  [294]
  [300]
  [306]
  [312]
  [318]
  [324]
  [330]
  [336]
  [342]
  [348]
  [354]
  [360]
  [366]
  [372]
  [378]
  [384]
  [390]
  [396]
  [402]
  [408]
  [414]
  [420]
  [426]
  [432]
  [438]
  [444]
  [450]
  [456]
  [462]
  [468]
  [474]
  [480]
  [486]
  [492]
  [498]
  [504]
  [510]
  [516]
  [522]
  [528]
  [534]
  [540]
  [546]
  [552]
  [558]
  [564]
  [570]
  [576]
  [582]
  [588]
  [594]
  [600]
  [606]
  [612]
  [618]
  [624]
  [630]
  [636]
  [642]
  [648]
  [654]
  [660]
  [666]]], shape=(1, 112, 1), dtype=int64)
tf.Tensor([1338], shape=(1,), dtype=int64)
